# Spreadsheet-Style Component Data Editor Demo

This notebook demonstrates the comprehensive Excel-like interface for editing multidimensional system dynamics data with dynamic dimension selection and climate adaptation features.

## Features Demonstrated

1. **Core Table Structure** - Excel-like spreadsheet interface
2. **Dynamic Dimension Selection** - Choose any 2 dimensions to display
3. **Visual Design** - Clean grid lines, alternating colors, proper headers
4. **Data Management** - Copy/paste, undo/redo, import/export
5. **Climate Adaptation** - Pre-configured scenarios and validation
6. **User Experience** - Keyboard shortcuts, context menus, search

## Requirements

```bash
pip install PyQt6 PyYAML numpy pandas matplotlib
```

In [ ]:
# Import required libraries
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd

# Add parent directory to path
sys.path.insert(0, str(Path().parent))

# Import PyQt6 for GUI
from PyQt6.QtWidgets import QApplication
from PyQt6.QtCore import Qt

# Check if we're in a Jupyter environment
def is_jupyter():
    try:
        from IPython import get_ipython
        return get_ipython() is not None
    except ImportError:
        return False

print(f"Running in Jupyter: {is_jupyter()}")
print(f"Python version: {sys.version}")

## 1. Basic Setup and Mock Data

First, let's create mock components and model data to demonstrate the spreadsheet editor.

In [ ]:
# Mock component class for demonstration
class MockComponent:
    """Mock component for testing the spreadsheet editor."""
    
    def __init__(self, name, component_type, spatial_dims=None):
        self.name = name
        self.component_type = component_type
        self.properties = {
            'spatial_dims': spatial_dims or ['parcel', 'time'],
            'description': f'Mock {component_type} component for testing',
            'units': 'units',
            'initial_value': 0
        }
        
    def __str__(self):
        return f"{self.component_type}({self.name})"


class MockModel:
    """Mock model for testing the spreadsheet editor."""
    
    def __init__(self):
        self.dimensions = {
            'parcel': {
                'labels': ['1001', '1002', '1003'],
                'description': 'Property parcel identifiers',
                'type': 'categorical',
                'size': 3
            },
            'time': {
                'labels': [str(year) for year in range(2020, 2031)],
                'description': 'Annual time steps',
                'type': 'temporal',
                'size': 11
            },
            'building_type': {
                'labels': ['single_family', 'multi_family', 'commercial'],
                'description': 'Building type categories',
                'type': 'categorical',
                'size': 3
            },
            'year_built_cohort': {
                'labels': ['pre_1950', '1950_1970', '1970_1990', '1990_2010', 'post_2010'],
                'description': 'Construction era cohorts',
                'type': 'categorical',
                'size': 5
            }
        }

# Create mock data
mock_model = MockModel()
mock_components = [
    MockComponent("basement_height_improvement_rate", "flow", ['building_type', 'year_built_cohort', 'parcel']),
    MockComponent("offset_improvement_rate", "flow", ['building_type', 'year_built_cohort', 'parcel']),
    MockComponent("structureBsmtHeight", "stock", ['parcel', 'time']),
    MockComponent("structureOffsetFromGrnd", "stock", ['parcel', 'time']),
    MockComponent("albedoChange", "parameter", ['parcel', 'time']),
    MockComponent("albedoTempInt", "parameter", ['parcel', 'time']),
    MockComponent("population", "stock", ['parcel', 'time']),
    MockComponent("housing_units", "stock", ['parcel', 'time'])
]

print(f"Created {len(mock_components)} mock components:")
for comp in mock_components:
    print(f"  - {comp.name} ({comp.component_type}) - dims: {comp.properties['spatial_dims']}")
    
print(f"\nAvailable dimensions: {list(mock_model.dimensions.keys())}")

## 2. Launch Spreadsheet Editor

Now let's launch the spreadsheet editor with our mock data. This will open a new window with the Excel-like interface.

In [ ]:
# Import the spreadsheet editor
try:
    from inspector.spreadsheet_editor import SpreadsheetDataEditor
    from inspector.climate_adaptation_features import ClimateAdaptationTemplates
    EDITOR_AVAILABLE = True
    print("✓ Spreadsheet editor imported successfully")
except ImportError as e:
    print(f"✗ Could not import spreadsheet editor: {e}")
    EDITOR_AVAILABLE = False

if EDITOR_AVAILABLE:
    # Create QApplication if it doesn't exist
    app = QApplication.instance()
    if app is None:
        app = QApplication(sys.argv)
    
    print("\nLaunching Spreadsheet Editor...")
    print("This will open a new window with the Excel-like interface.")
    print("\nFeatures to try:")
    print("1. Use the dimension selectors to choose different dimension combinations")
    print("2. Click on cells to edit data directly")
    print("3. Use Ctrl+C/Ctrl+V to copy and paste data")
    print("4. Try the 'Apply Template' button for quick data patterns")
    print("5. Use 'Climate Presets' for pre-configured scenarios")
    print("6. Export data to CSV or import from external files")
    
    # Launch editor with flood protection components
    flood_components = [
        comp for comp in mock_components 
        if comp.name in ['basement_height_improvement_rate', 'offset_improvement_rate', 
                       'structureBsmtHeight', 'structureOffsetFromGrnd']
    ]
    
    editor = SpreadsheetDataEditor(flood_components, mock_model)
    editor.show()
    
    print(f"\n✓ Editor launched with {len(flood_components)} flood protection components")
else:
    print("\n✗ Cannot launch editor - missing dependencies")
    print("Please install: pip install PyQt6 PyYAML numpy")

## 3. Climate Adaptation Scenarios

Let's demonstrate the climate adaptation specific features by creating different scenario editors.

In [ ]:
if EDITOR_AVAILABLE:
    # Show available climate templates
    templates = ClimateAdaptationTemplates.get_dimension_combinations()
    
    print("Available Climate Adaptation Templates:")
    print("=" * 40)
    
    for key, template in templates.items():
        print(f"\n{template['name']}:")
        print(f"  Dimensions: {' × '.join(template['dimensions'])}")
        print(f"  Description: {template['description']}")
        print(f"  Variables: {', '.join(template['variables'])}")
    
    print("\n" + "=" * 40)
    
    # Launch albedo changes scenario
    print("\nLaunching Albedo Changes Scenario...")
    
    albedo_components = [
        comp for comp in mock_components 
        if comp.name in ['albedoChange', 'albedoTempInt']
    ]
    
    albedo_editor = SpreadsheetDataEditor(albedo_components, mock_model)
    albedo_editor.setWindowTitle("Albedo Changes Scenario - Spreadsheet Editor")
    albedo_editor.show()
    
    print(f"✓ Albedo scenario launched with {len(albedo_components)} components")
    print("\nTry using the 'Climate Presets' button to apply the albedo changes template!")
else:
    print("Editor not available - cannot demonstrate climate scenarios")

## 4. Data Pattern Generation

Let's demonstrate the data pattern generation capabilities for common climate modeling scenarios.

In [ ]:
if EDITOR_AVAILABLE:
    from inspector.climate_adaptation_features import DataPatternGenerator
    import matplotlib.pyplot as plt
    
    # Generate sample data patterns
    years = list(range(2020, 2051))
    
    # Linear growth pattern (e.g., gradual improvement in flood protection)
    linear_pattern = DataPatternGenerator.linear_growth(0.1, 0.8, len(years))
    
    # Exponential growth pattern (e.g., accelerating adaptation measures)
    exponential_pattern = DataPatternGenerator.exponential_growth(0.1, 0.05, len(years))
    
    # Climate scenario ramp (e.g., increasing climate impacts)
    climate_ramp = DataPatternGenerator.climate_scenario_ramp(
        baseline=100,  # baseline value
        impact_factor=0.3,  # 30% increase by end
        start_year=2030,  # impacts start in 2030
        end_year=2050,  # full impact by 2050
        current_years=years
    )
    
    # Plot the patterns
    plt.figure(figsize=(12, 8))
    
    plt.subplot(2, 2, 1)
    plt.plot(years, linear_pattern, 'b-', linewidth=2)
    plt.title('Linear Growth Pattern\n(Gradual Improvement)')
    plt.xlabel('Year')
    plt.ylabel('Value')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(2, 2, 2)
    plt.plot(years, exponential_pattern, 'g-', linewidth=2)
    plt.title('Exponential Growth Pattern\n(Accelerating Adaptation)')
    plt.xlabel('Year')
    plt.ylabel('Value')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(2, 2, 3)
    plt.plot(years, climate_ramp, 'r-', linewidth=2)
    plt.title('Climate Scenario Ramp\n(Increasing Impacts)')
    plt.xlabel('Year')
    plt.ylabel('Value')
    plt.grid(True, alpha=0.3)
    
    # Seasonal variation (monthly data for one year)
    months = list(range(1, 13))
    seasonal_pattern = DataPatternGenerator.seasonal_variation(50, 20, 12)
    
    plt.subplot(2, 2, 4)
    plt.plot(months, seasonal_pattern, 'purple', linewidth=2, marker='o')
    plt.title('Seasonal Variation Pattern\n(Monthly Fluctuations)')
    plt.xlabel('Month')
    plt.ylabel('Value')
    plt.grid(True, alpha=0.3)
    plt.xticks(months)
    
    plt.tight_layout()
    plt.show()
    
    print("\nData Pattern Examples:")
    print(f"Linear Growth: {linear_pattern[0]:.2f} → {linear_pattern[-1]:.2f}")
    print(f"Exponential Growth: {exponential_pattern[0]:.2f} → {exponential_pattern[-1]:.2f}")
    print(f"Climate Ramp: {climate_ramp[0]:.2f} → {climate_ramp[-1]:.2f}")
    print(f"Seasonal Range: {min(seasonal_pattern):.2f} to {max(seasonal_pattern):.2f}")
    
    print("\nThese patterns can be applied to selected cells using the 'Apply Template' feature!")
else:
    print("Editor not available - cannot demonstrate data patterns")

## 5. Multi-Component Editor

Finally, let's launch a comprehensive editor with all components to demonstrate the full capabilities.

In [ ]:
if EDITOR_AVAILABLE:
    print("Launching Comprehensive Multi-Component Editor...")
    print("\nThis editor includes all climate adaptation components:")
    
    for i, comp in enumerate(mock_components, 1):
        dims_str = ' × '.join(comp.properties['spatial_dims'])
        print(f"  {i}. {comp.name} ({comp.component_type}) - {dims_str}")
    
    # Launch comprehensive editor
    comprehensive_editor = SpreadsheetDataEditor(mock_components, mock_model)
    comprehensive_editor.setWindowTitle("Comprehensive Climate Adaptation - Spreadsheet Editor")
    comprehensive_editor.show()
    
    print(f"\n✓ Comprehensive editor launched with {len(mock_components)} components")
    
    print("\nFeatures to explore:")
    print("1. Switch between different dimension combinations")
    print("2. Use climate presets for quick setup")
    print("3. Apply data templates to generate realistic patterns")
    print("4. Validate data using climate-specific rules")
    print("5. Export/import data for external analysis")
    print("6. Copy/paste data between different dimension views")
    
    print("\n" + "=" * 60)
    print("SPREADSHEET EDITOR DEMO COMPLETE")
    print("=" * 60)
    print("\nThe spreadsheet editors are now running in separate windows.")
    print("You can interact with them to explore all the features.")
    print("\nClose the editor windows when you're done exploring.")
else:
    print("\n✗ Cannot launch comprehensive editor")
    print("Please ensure all dependencies are installed and try again.")

## Summary

This demo has shown the comprehensive spreadsheet-style component data editor with:

### ✅ Core Features Implemented
- **Excel-like Interface**: Clean grid lines, alternating row colors, resizable columns
- **Dynamic Dimensions**: Select any 2 dimensions from available options
- **Direct Cell Editing**: Click and edit values with real-time validation
- **Copy/Paste Support**: Standard keyboard shortcuts for data manipulation
- **Import/Export**: CSV support for external data integration
- **Undo/Redo**: Full history tracking for data changes

### 🌍 Climate Adaptation Features
- **Pre-configured Templates**: Flood protection, albedo changes, building cohorts
- **Validation Rules**: Climate-specific data validation
- **Data Patterns**: Linear growth, exponential, seasonal, climate ramps
- **Parcel-based Structure**: Support for property-level modeling
- **Time Series Support**: Annual, decadal, and climate period dimensions

### 🎯 Integration Benefits
- **Replaces Old Editor**: Modern interface instead of basic multidimensional editor
- **YAML Compatible**: Maintains compatibility with existing model definitions
- **GUI Integration**: Seamless integration with component inspector
- **Model Updates**: Changes are properly saved and reflected in simulations

The spreadsheet editor provides an intuitive, familiar interface for climate adaptation researchers and modelers to input, view, and modify complex multidimensional data while maintaining the flexibility to work with any combination of dimensions.